# 06 - Proyek Akhir Kursus Coursera: Regresi

> Ini proyek akhir **kursus Coursera** (Modul 6, "Tugas Akhir"), bukan topik TA mata kuliah.

Instruksi tugas: terapkan teknik regresi untuk menganalisis dataset pilihan. Lakukan **prapemrosesan data**, **bangun dan bandingkan model**, **ekstrak wawasan (insight)**, dan **sarankan langkah selanjutnya**. Laporan berupa ringkasan jelas tentang proses dan kesimpulan.

## 1. Latar Belakang & Rumusan Masalah
(...)

## 2. Dataset
(sumber, jumlah data, fitur, target) - contoh di bawah memakai Diabetes; ganti dengan dataset pilihanmu.

## 3. Metodologi
(Prapemrosesan -> Split data -> Modeling (Linear, Polynomial, Ridge, Lasso, Elastic Net) -> Validasi silang -> Evaluasi -> Insight)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, KFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

sns.set_theme(style='whitegrid')

# Ganti path dan nama target kalau pakai dataset lain
df = pd.read_csv('../datasets/diabetes.csv')
TARGET = 'progres_penyakit'
df.head()

## 4. Eksplorasi & Prapemrosesan

In [ ]:
print(df.shape)
df.info()
print('Missing value:', df.isna().sum().sum())
print('Data duplikat:', df.duplicated().sum())
df.describe().T

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(df[TARGET], kde=True, ax=ax[0]); ax[0].set_title('Distribusi target')
sns.heatmap(df.corr(), cmap='coolwarm', center=0, annot=True, fmt='.2f', annot_kws={'size': 7}, ax=ax[1])
ax[1].set_title('Korelasi')
plt.tight_layout(); plt.show()

In [ ]:
# Pencilan (IQR) pada fitur
X = df.drop(columns=TARGET)
Q1, Q3 = X.quantile(0.25), X.quantile(0.75)
IQR = Q3 - Q1
outlier = ((X < Q1 - 1.5 * IQR) | (X > Q3 + 1.5 * IQR)).sum()
print('Jumlah pencilan per fitur:')
print(outlier[outlier > 0])

# Pisahkan data
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape)

## 5. Modeling: bandingkan beberapa model dengan validasi silang

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

def poly(derajat, model):
    return make_pipeline(PolynomialFeatures(derajat, include_bias=False), StandardScaler(), model)

kandidat = {
    'Linear':              make_pipeline(StandardScaler(), LinearRegression()),
    'Polynomial deg 2':    poly(2, LinearRegression()),
    'Ridge':               make_pipeline(StandardScaler(), Ridge(alpha=10)),
    'Lasso':               make_pipeline(StandardScaler(), Lasso(alpha=1, max_iter=10000)),
    'ElasticNet':          make_pipeline(StandardScaler(), ElasticNet(alpha=1, l1_ratio=0.5, max_iter=10000)),
    'Polynomial deg 2 + Ridge': poly(2, Ridge(alpha=100)),
}

hasil = []
for nama, m in kandidat.items():
    rmse_cv = np.sqrt(-cross_val_score(m, X_train, y_train, cv=kf, scoring='neg_mean_squared_error'))
    r2_cv = cross_val_score(m, X_train, y_train, cv=kf, scoring='r2')
    hasil.append([nama, rmse_cv.mean(), rmse_cv.std(), r2_cv.mean()])

tabel = pd.DataFrame(hasil, columns=['Model', 'RMSE CV', 'Std RMSE', 'R2 CV']).sort_values('RMSE CV').round(3)
tabel

### Tuning hyperparameter model terbaik (GridSearchCV)

In [ ]:
pipe = make_pipeline(StandardScaler(), Ridge())
grid = GridSearchCV(pipe, {'ridge__alpha': np.logspace(-2, 3, 30)}, cv=kf, scoring='neg_mean_squared_error')
grid.fit(X_train, y_train)
print('Alpha Ridge terbaik:', round(grid.best_params_['ridge__alpha'], 3))
print('RMSE CV            :', round(np.sqrt(-grid.best_score_), 2))

## 6. Evaluasi pada data test

In [ ]:
final = grid.best_estimator_
pred = final.predict(X_test)

print('RMSE:', round(np.sqrt(mean_squared_error(y_test, pred)), 2))
print('MAE :', round(mean_absolute_error(y_test, pred), 2))
print('R2  :', round(r2_score(y_test, pred), 4))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].scatter(y_test, pred, alpha=0.6)
ax[0].plot([y.min(), y.max()], [y.min(), y.max()], 'r--')
ax[0].set_xlabel('Asli'); ax[0].set_ylabel('Prediksi'); ax[0].set_title('Prediksi vs Asli')
ax[1].scatter(pred, y_test - pred, alpha=0.6)
ax[1].axhline(0, color='red')
ax[1].set_xlabel('Prediksi'); ax[1].set_ylabel('Residual'); ax[1].set_title('Residual')
plt.tight_layout(); plt.show()

## 7. Insight: fitur apa yang paling berpengaruh?

In [ ]:
koef = pd.Series(final[-1].coef_, index=X.columns).sort_values()
koef.plot(kind='barh', color=['tomato' if k < 0 else 'steelblue' for k in koef])
plt.title('Koefisien model akhir (data ter-scale)')
plt.show()
koef.round(2).sort_values(key=abs, ascending=False)

## 8. Kesimpulan & Langkah Selanjutnya
- Model terbaik: (...)
- Temuan utama / fitur berpengaruh: (...)
- Keterbatasan: (...)
- Langkah selanjutnya: (misal coba model non-linier, tambah fitur, kumpulkan data lebih banyak, dll.)